# Text2Cypher Toolkit: Full Pipeline

This notebook executes the complete end-to-end machine learning pipeline for the Engineering Capstone project. It covers:
1. **Environment Setup:** Cloning the repository, loading secrets, and installing dependencies.
2. **Download Data:** Fetching the dataset and model weights from the Hugging Face Hub.
3. **Diagnostics:** Checking hardware constraints (T4 GPU), package dependencies, and previewing the dataset.
4. **Fine-Tuning:** Executing LoRA training on lightweight LLMs (Gemma/Qwen).
5. **Inference (Prediction):** Generating Cypher queries using the fine-tuned adapter.
6. **Evaluation:** Scoring outputs on translation (GLEU) and execution (Exact Match) metrics.
7. **Exporting (GGUF):** Merging the adapter and compiling for local edge-device execution via `llama.cpp`.

### ⚠️ Prerequisites
Ensure you have added your Hugging Face Access Token as a Colab Secret named `HF_TOKEN` (with **Write** permissions) before running the cells below.

## 1. Environment Setup & Bootstrap

In [ ]:
# Clone the repository and navigate into it
!git clone https://github.com/Cxmrykk/capstone.git /content/capstone
%cd /content/capstone

# Configure Environment Variables
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded successfully from Colab Secrets.")
except Exception:
    print("❌ ERROR: Please set the 'HF_TOKEN' secret in Google Colab (with write permissions).")

# Target Hugging Face Hub repository for remote checkpoint sync
os.environ["T2C_HUB_REPO"] = "2l3c34o1/text2cypher-checkpoints"

# Run the bootstrap script to install PyTorch, Transformers, Unsloth, etc.
!bash scripts/colab_bootstrap.sh

## 2. Download Dataset & Models
Fetch the `text2cypher-2024v1` dataset and `gemma-4-E2B` model weights directly from the Hugging Face Hub to the local filesystem.

In [ ]:
# Fetch the dataset and model weights via the provided script
!bash download_data.sh

## 3. System Diagnostics & Data Preview
Verify that the T4 GPU is active, packages are installed correctly, and inspect how the prompt filtering reduces token bloat.

In [ ]:
# Verify GPU, CUDA, and library availability
!python app.py doctor

# Preview a formatted Prompt-Target example to observe schema trimming behaviour
!python app.py data preview --config configs/gemma-4-e2b.yaml --index 5

## 4. Model Fine-Tuning (LoRA)
Run the training loop using the designated parameter-efficient fine-tuning configuration. The current setup is mapped to `gemma-4-e2b`.

In [ ]:
# Train the model.
!python app.py train --config configs/gemma-4-e2b.yaml

## 5. Inference (Cypher Generation)
Generate Cypher query predictions on the holdout test split. We'll run inference on a subset of 100 queries to save time, using the adapter trained in the previous step.

In [ ]:
!python app.py predict \
    --config configs/gemma-4-e2b.yaml \
    --adapter artifacts/runs/gemma-4-e2b-enhanced/final \
    --split test \
    --limit 100 \
    --out artifacts/predictions/gemma-4-e2b-test.jsonl

## 6. Evaluation
Score the model outputs across syntactic validity, translation fidelity (Google-BLEU), and logical execution against the remote Neo4j Labs demo databases.

In [ ]:
!python app.py evaluate \
    --predictions artifacts/predictions/gemma-4-e2b-test.jsonl \
    --execute \
    --validate-syntax

## 7. Export to Edge Format (GGUF)
Once satisfied with the model's performance, merge the LoRA weights back into the base model and quantize it to a 4-bit `GGUF` file. This format allows the model to be run quickly and cheaply on local CPU/Edge hardware.

In [ ]:
# Step 1: Merge LoRA adapter into full-precision base weights
!python app.py export merge \
    --config configs/gemma-4-e2b.yaml \
    --adapter artifacts/runs/gemma-4-e2b-enhanced/final \
    --out artifacts/merged/gemma-4-e2b-enhanced

# Step 2: Build llama.cpp with CUDA support
!BUILD_CUDA=1 bash scripts/build_llama_cpp.sh

# Step 3: Quantize the merged weights to Q4_K_M
!python app.py export gguf \
    --merged artifacts/merged/gemma-4-e2b-enhanced \
    --quant Q4_K_M